##1.	Exploratory Analysis (30%):
###1)	Distributional analysis, subgroup pattern exploration, and hypothesis generation (minimum 8–10 figures)


In [0]:
import matplotlib.pyplot as mp
import pandas as pd
import seaborn as sb
import numpy as np
import random
from datetime import datetime

In [0]:
df_spark = spark.read.table("workspace.default.SP500_with_indicators")

df = df_spark.toPandas()

In [0]:
df.describe()

In [0]:
df.dtypes

## 1) Distribution Analysis

In [0]:
def histogram(x):
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    color = random.choice(colors)
    mp.hist(df[x], color=color, ec='black', bins=15)
    mp.xlabel(x)
    mp.ylabel("Amount")
    mp.title('Distribution of SPX ' + x)
    mp.show()

In [0]:
for column in df:
    histogram(column)

In [0]:
mp.boxplot(df['Volume'])
mp.title("Volume Boxplot")
mp.show()

###2) Subgroup Pattern Exploration

In [0]:
def scatter(x, y, color):
    mp.scatter(df[x], df[y], color=color, marker='o', s=100, alpha=0.7)
    mp.xlabel(x)
    mp.ylabel(y)
    mp.title(f'{y} vs {x}')
    mp.show()

In [0]:
scatter('SMA_200','ret_1d', 'green')

In [0]:
date_column = "Date"
price_column = "Close"

#Specify range
start_date = pd.to_datetime("2021-01-04")
end_half_date = pd.to_datetime("2023-07-04")
end_date = pd.to_datetime("2025-12-31")


df[date_column] = pd.to_datetime(df[date_column], errors='coerce')

mask = (df[date_column] >= start_date) & (df[date_column] <= end_half_date)

mask2 = (df[date_column] > end_half_date) & (df[date_column] <= end_date)

filtered_df = df.loc[mask]

filtered_df2 = df.loc[mask2]

if filtered_df.empty or filtered_df2.empty:
    print("No data found in date range")
else:
    mp.figure(figsize=(10,6))
    mp.plot(filtered_df[date_column], filtered_df[price_column], marker='o', linestyle='-')
    mp.title(f"Prices from {start_date} to {end_half_date}")
    mp.xlabel("Date")
    mp.ylabel("Price($)")
    mp.grid(True)
    mp.tight_layout()
    mp.show()

    mp.figure(figsize=(10,6))
    mp.plot(filtered_df2[date_column], filtered_df2[price_column], marker='o', linestyle='-')
    mp.title(f"Prices from {end_half_date} to {end_date}")
    mp.xlabel("Date")
    mp.ylabel("Price($)")
    mp.grid(True)
    mp.tight_layout()
    mp.show()

In [0]:
# Make sure Date is datetime + sorted
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values("Date")

# Regime label (Bull if above SMA_200)
df["Regime"] = (df["Close"] >= df["SMA_200"]).map({True: "Bull", False: "Bear"})

# Boolean masks for shading
is_bull = (df["Regime"] == "Bull").fillna(False)
is_bear = (df["Regime"] == "Bear").fillna(False)

ymin = df["Close"].min()
ymax = df["Close"].max()

mp.figure(figsize=(12, 6))
mp.plot(df["Date"], df["Close"], label="Close")
mp.plot(df["Date"], df["SMA_200"], label="SMA_200")

mp.fill_between(df["Date"], ymin, ymax, where=is_bull, alpha=0.15, label = "Bullish")
mp.fill_between(df["Date"], ymin, ymax, where=is_bear, alpha=0.15, label = "Bearish")

mp.title("Bull vs Bear Regimes (Close vs Simple Moving Average 200 Days(SMA_200))")
mp.xlabel("Date")
mp.ylabel("Price ($)")
mp.legend(loc = "lower right")
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
df_spark1 = spark.read.table("workspace.default.capstonetrades_closesma200")

df_spark2 = spark.read.table("workspace.default.capstonetrades_sma20_50")

df_spark3 = spark.read.table("workspace.default.capstonetrades_rsi40")

df_spark4 = spark.read.table("workspace.default.capstonetrades_sma20_50_200")


df_CloseSMA200 = df_spark1.toPandas()

df_SMA2050 = df_spark2.toPandas()

df_RSI40 = df_spark3.toPandas()

df_SMA2050200 = df_spark4.toPandas()

In [0]:
pdf = pd.concat([df_CloseSMA200, df_SMA2050, df_RSI40, df_SMA2050200], ignore_index=True)

pdf.head


In [0]:
pdf = pd.concat([df_CloseSMA200, df_SMA2050, df_RSI40, df_SMA2050200], ignore_index=True)

# Clean and sort
pdf["exit_date"] = pd.to_datetime(pdf["exit_date"])
pdf = pdf.sort_values(["strategy_id", "exit_date"])

# Plot
mp.figure(figsize=(12, 6))

for strategy, group in pdf.groupby("strategy_id"):
    mp.plot(group["exit_date"], group["cumulative_profit"], label=strategy)

mp.title("Cumulative Profit Over Time by Strategy")
mp.xlabel("Date")
mp.ylabel("Cumulative Profit")
mp.legend(title="Strategy")
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
# Count trades per strategy
trade_counts = pdf.groupby("strategy_id").size()

print(trade_counts)

# Plot
mp.figure(figsize=(8,5))
trade_counts.plot(kind="bar")

mp.title("Total Trades per Strategy")
mp.xlabel("Strategy")
mp.ylabel("Number of Trades")
mp.grid(axis="y", linestyle="--", alpha=0.5)

mp.tight_layout()
mp.show()

In [0]:
# Make sure dates are datetime
df["Date"] = pd.to_datetime(df["Date"])
df_SMA2050["entry_date"] = pd.to_datetime(df_SMA2050["entry_date"])

# Make sure sorted
df = df.sort_values("Date")
trades_df = df_SMA2050.sort_values("entry_date")

mp.figure(figsize=(14, 7))

# Main price line
mp.plot(df["Date"], df["Close"], label="Close")

# VWAP line
mp.plot(df["Date"], df["VWAP"], label="VWAP")

# Entry markers
mp.scatter(
    trades_df["entry_date"],
    trades_df["entry_price"],   # best if you already stored entry_price
    marker="^",
    s=80,
    label="Entry"
)

mp.title("Price, VWAP, and Strategy Entry Dates")
mp.xlabel("Date")
mp.ylabel("Price")
mp.legend()
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
# Make sure dates are datetime
df["Date"] = pd.to_datetime(df["Date"])
df_SMA2050200["entry_date"] = pd.to_datetime(df_SMA2050200["entry_date"])

# Make sure sorted
df = df.sort_values("Date")
trades_df = df_SMA2050200.sort_values("entry_date")

mp.figure(figsize=(14, 7))

# Main price line
mp.plot(df["Date"], df["Close"], label="Close")

# VWAP line
mp.plot(df["Date"], df["VWAP"], label="VWAP")

# Entry markers
mp.scatter(
    trades_df["entry_date"],
    trades_df["entry_price"],   # best if you already stored entry_price
    marker="^",
    s=80,
    label="Entry"
)

mp.title("Price, VWAP, and Strategy Entry Dates")
mp.xlabel("Date")
mp.ylabel("Price")
mp.legend()
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
# Make sure dates are datetime
df["Date"] = pd.to_datetime(df["Date"])
df_RSI40["entry_date"] = pd.to_datetime(df_RSI40["entry_date"])

# Make sure sorted
df = df.sort_values("Date")
trades_df = df_RSI40.sort_values("entry_date")

mp.figure(figsize=(14, 7))

# Main price line
mp.plot(df["Date"], df["Close"], label="Close")

# VWAP line
mp.plot(df["Date"], df["VWAP"], label="VWAP")

# Entry markers
mp.scatter(
    trades_df["entry_date"],
    trades_df["entry_price"],   # best if you already stored entry_price
    marker="^",
    s=80,
    label="Entry"
)

mp.title("Price, VWAP, and Strategy Entry Dates")
mp.xlabel("Date")
mp.ylabel("Price")
mp.legend()
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
# Make sure dates are datetime
df["Date"] = pd.to_datetime(df["Date"])
df_CloseSMA200["entry_date"] = pd.to_datetime(df_CloseSMA200["entry_date"])

# Make sure sorted
df = df.sort_values("Date")
trades_df = df_CloseSMA200.sort_values("entry_date")

mp.figure(figsize=(14, 7))

# Main price line
mp.plot(df["Date"], df["Close"], label="Close")

# VWAP line
mp.plot(df["Date"], df["VWAP"], label="VWAP")

# Entry markers
mp.scatter(
    trades_df["entry_date"],
    trades_df["entry_price"],   # best if you already stored entry_price
    marker="^",
    s=80,
    label="Entry"
)

mp.title("Price, VWAP, and Strategy Entry Dates")
mp.xlabel("Date")
mp.ylabel("Price")
mp.legend()
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:

# Make sure dates are datetime
df["Date"] = pd.to_datetime(df["Date"])
pdf["entry_date"] = pd.to_datetime(pdf["entry_date"])
pdf["exit_date"] = pd.to_datetime(pdf["exit_date"])

# Sort data
df = df.sort_values("Date")
pdf = pdf.sort_values("entry_date")

# --- Get entry prices from main df if not already in trades_df
entries_plot = pdf.merge(
    df[["Date", "Close"]],
    left_on="entry_date",
    right_on="Date",
    how="left"
)

exits_plot = pdf.merge(
    df[["Date", "Close"]],
    left_on="exit_date",
    right_on="Date",
    how="left"
)

mp.figure(figsize=(16, 8))

# Main lines
mp.plot(df["Date"], df["Close"], label="Close", linewidth=1.5)
mp.plot(df["Date"], df["VWAP"], label="VWAP", linewidth=1.5)

# Plot each strategy separately
for strat in pdf["strategy_id"].dropna().unique():
    strat_entries = entries_plot[entries_plot["strategy_id"] == strat]
    strat_exits = exits_plot[exits_plot["strategy_id"] == strat]

    mp.scatter(
        strat_entries["entry_date"],
        strat_entries["Close"],
        s=70,
        marker="^",
        label=f"{strat} Entry"
    )

    mp.scatter(
        strat_exits["exit_date"],
        strat_exits["Close"],
        s=70,
        marker="v",
        label=f"{strat} Exit"
    )

mp.title("Close, VWAP, and Trade Signals by Strategy")
mp.xlabel("Date")
mp.ylabel("Price")
mp.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
mp.grid(True)
mp.tight_layout()
mp.show()

In [0]:
import pandas as pd

# Market data
df = df_spark.toPandas().copy()
df["Date"] = pd.to_datetime(df["Date"])


# Read strategy trade tables
df_spark1 = spark.read.table("workspace.default.capstonetrades_closesma200_v2")
df_spark2 = spark.read.table("workspace.default.capstonetrades_sma20_50_v2")
df_spark3 = spark.read.table("workspace.default.capstonetrades_rsi40_v2")
df_spark4 = spark.read.table("workspace.default.capstonetrades_sma20_50_200_v2")

df_CloseSMA200 = df_spark1.toPandas().copy()
df_SMA2050 = df_spark2.toPandas().copy()
df_RSI40 = df_spark3.toPandas().copy()
df_SMA2050200 = df_spark4.toPandas().copy()


# Helper function
def prepare_trade_table(trades_df, strategy_name):
    trades_df = trades_df.copy()

    # Standardize types
    trades_df["entry_date"] = pd.to_datetime(trades_df["entry_date"])
    trades_df["exit_date"] = pd.to_datetime(trades_df["exit_date"])
    trades_df["pnl_price"] = pd.to_numeric(trades_df["pnl_price"], errors="coerce")

    # Add strategy + profitability label
    trades_df["strategy"] = strategy_name
    trades_df["profitable"] = (trades_df["pnl_price"] > 0).astype(int)

    return trades_df


# Prepare each strategy table

trades_1 = prepare_trade_table(df_CloseSMA200, "CloseSMA200_V2")
trades_2 = prepare_trade_table(df_SMA2050, "SMA20_50_C2")
trades_3 = prepare_trade_table(df_RSI40, "RSI40_V2")
trades_4 = prepare_trade_table(df_SMA2050200, "SMA20_50_200_V2")


# Combine all trades
trades_all = pd.concat(
    [trades_1, trades_2, trades_3, trades_4],
    ignore_index=True
)

# Keep a cleane trading dataset
trade_labels = trades_all[[
    "entry_date",
    "exit_date",
    "strategy",
    "pnl_price",
    "profitable"
]].copy()


# Merge tables
df_model = df.merge(
    trade_labels,
    left_on="Date",
    right_on="entry_date",
    how="left"
)

df_model["strategy"] = df_model["strategy"].fillna("NoTrade")
df_model["trade_entry"] = (df_model["strategy"] != "NoTrade").astype(int)

# Keep profitable as missing on non entry days
df_model["profitable"] = df_model["profitable"].fillna(0).astype(int)
df_model["pnl_price"] = df_model["pnl_price"].fillna(0)


# Create a trade only table / dataset
trade_model = trade_labels.merge(
    df,
    left_on="entry_date",
    right_on="Date",
    how="left"
)


# Convert Pandas back to Spark
trade_labels_spark = spark.createDataFrame(trade_labels)
df_model_spark = spark.createDataFrame(df_model)
trade_model_spark = spark.createDataFrame(trade_model)


# Save as Delta tables
trade_labels_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.capstone_trade_labels_V2"
)

df_model_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.capstone_daily_model_data_V2"
)

trade_model_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.capstone_trade_model_data_V2"
)


# Table check
print(trade_labels.head())
print(df_model.head())
print(trade_model.head())

In [0]:
# Make sure Date is datetime + sorted
df_spark = spark.read.table("workspace.default.SP500_with_indicators")

df = df_spark.toPandas()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values("Date")

# Regime label (Bull if above SMA_200)
df["Regime"] = (df["Close"] >= df["SMA_200"]).map({True: "Bull", False: "Bear"})

# Boolean masks for shading
is_bull = (df["Regime"] == "Bull").fillna(False)
is_bear = (df["Regime"] == "Bear").fillna(False)

ymin = df["Close"].min()
ymax = df["Close"].max()

mp.figure(figsize=(12, 6))
mp.plot(df["Date"], df["Close"], label="Close")
mp.plot(df["Date"], df["SMA_200"], label="SMA_200")

mp.fill_between(df["Date"], ymin, ymax, where=is_bull, alpha=0.15, label = "Bullish")
mp.fill_between(df["Date"], ymin, ymax, where=is_bear, alpha=0.15, label = "Bearish")

mp.title("Bull vs Bear Regimes (Close vs Simple Moving Average 200 Days(SMA_200))")
mp.xlabel("Date")
mp.ylabel("Price ($)")
mp.legend(loc = "lower right")
mp.grid(True)
mp.tight_layout()
mp.show()